In [1]:
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from astropy import units as u
from astropy import constants as c
import rebound
import seaborn as sns
from matplotlib import rcParams

sol_to_earth = 332946.078
AU = 1.496e+13 #cm
M_sun = 1.989e+33 #g
G = 6.6743* 10**(-8) #cm^-3.g^-1.s^-2

r_emb_chambers = 2.39e-5 * AU #cm, radius of embryo
r_pl_chambers = 1.11e-5 * AU #cm, radius of planetesimal

In [2]:
def get_r(m, rho):
    return (m/(4/3 * np.pi * rho))**(1/3)

r_earth = get_r(1 * u.M_earth.to(u.g), 5.5) * u.cm.to(u.R_earth)
print("test, r_earth (in R_EARTH) = ", r_earth)

test, r_earth (in R_EARTH) =  0.9996982038103556


In [3]:
m_t = 360 * 1e-8 * u.solMass.to(u.g) #g
m_p = 10 * 0.5e-8 * u.solMass.to(u.g) #g
rho = 5.5 #g/cm^3
r_t = get_r(m_t, rho) #cm
r_p = get_r (m_p, rho) #cm
m_init = m_t + m_p
r_tot = r_t + r_p
theta = np.pi/3
b = r_tot * np.sin(theta)
mu = (m_t * m_p)/(m_t + m_p)
l = r_tot - b
l = min(l, 2*r_p)
v_imp = 30 * u.AU.to(u.cm) / (u.year.to(u.s)) #cm/s

alpha = (l**2 * (3*r_p - l))/(4 * r_p**3)
alpha = min(1, alpha)

q = 1/2 * v_imp**2 * m_t * m_p /(m_init**2)
v_esc = np.sqrt(2 * G * m_init / r_tot)
alpha_mu = (alpha * m_t * m_p)/(alpha * m_p + m_t)
gamma = m_p/m_t

cstar = 1.8
rho1 = 1 #g/cm^3
Rc1 = ( (3 / (4 * np.pi * rho1)) * m_init)**(1/3)
q_zero = 0.8 * cstar * np.pi * rho1 * G * Rc1**2
q_star = (mu/alpha_mu)**(1.5) * (1+gamma)**2/(4 * gamma) * q_zero
q_ratio = q/q_star
print("q/q_star = ", q_ratio)
print("b/r_t = ", b/r_t)
if (q_ratio < 1.8):
    Mlr = m_init * (1 - 0.5 * q_ratio)
else:
    Mlr = 0.1 * m_init * (q_ratio/1.8)**(-1.5)

print("Mlr/Mt = ", Mlr/m_t)



q/q_star =  0.02493081286688905
b/r_t =  1.0741961981983397
Mlr/Mt =  1.0012503518105353


In [4]:
phi = 2 * np.arccos((l - r_p) / r_p)
A_interact = (r_p**2) * (np.pi - (phi - np.sin(phi)) / 2.0)
L_interact = 2.0 * np.sqrt(r_t**2 - (r_t - l / 2.0)**2)
beta = (A_interact * L_interact * rho) / m_t
m_interact = A_interact * L_interact * rho
Rc1_g = ((3.0 / (4.0 * np.pi * rho1)) * (beta * m_t + m_p))**(1.0 / 3.0)
Q0_g = 0.8 * cstar * np.pi * rho1 * G * Rc1_g**2
gamma_g = beta * m_t / m_p
Q_star_g = ((1 + gamma_g)**2 / (4 * gamma_g)) * Q0_g
mu_g = (beta * m_t * m_p) / (beta * m_t + m_p)
Q_g = 0.5 * mu_g * v_imp**2 / (beta * m_t + m_p)

Q_g_ratio = Q_g/Q_star_g

if Q_g_ratio < 1.8:
    Mslr = (beta * m_t + m_p) * (1 - Q_g/(2 * Q_star_g))
else:
    Mslr = (beta * m_t + m_p)/10 * (Q_g/(Q_star_g * 1.8))**(-1.5)

print("q/q_star g = ", Q_g_ratio)
print("Mslr/Mp = ", Mslr/m_p)

q/q_star g =  669.7855079356638
Mslr/Mp =  2.457171402698699e-05


In [5]:
print("M_t (solMass), R_t (AU) = ", m_t * u.g.to(u.solMass), r_t * u.cm.to(u.AU))
print("M_p (solMass), R_p (AU) = ", m_p * u.g.to(u.solMass), r_p * u.cm.to(u.AU))

print("total init mass (solMass) = ", m_init * u.g.to(u.solMass))
print("Mlr = ", Mlr * u.g.to(u.solMass))
print("Mslr = ", Mslr * u.g.to(u.solMass))
print("Mtot - Mlr - Mslr = ", (m_init - Mlr - Mslr) * u.g.to(u.solMass))

M_t (solMass), R_t (AU) =  3.6000000000000003e-06 4.5275191098009734e-05
M_p (solMass), R_p (AU) =  5e-08 1.0883020817781722e-05
total init mass (solMass) =  3.65e-06
Mlr =  3.6045012665179277e-06
Mslr =  1.2285857013493495e-12
Mtot - Mlr - Mslr =  4.54975048963713e-08


In [6]:
(r_t + b) * u.cm.to(u.AU)

np.float64(9.390962924819509e-05)

In [7]:
(r_t + r_p) * np.sin(theta) * u.cm.to(u.AU)

np.float64(4.8634438150185366e-05)